# Lab 7 — The Four Fundamental Subspaces
**Linear Algebra · UATX**

For $A \in \mathbb{R}^{m \times n}$ of rank $r$:

| Subspace | Notation | Ambient | Dimension |
|---|---|---|---|
| Column space | $\mathrm{col}(A)$ | $\mathbb{R}^m$ | $r$ |
| Null space | $\ker(A)$ | $\mathbb{R}^n$ | $n-r$ |
| Row space | $\mathrm{row}(A) = \mathrm{col}(A^\top)$ | $\mathbb{R}^n$ | $r$ |
| Left null space | $\ker(A^\top)$ | $\mathbb{R}^m$ | $m-r$ |

**Orthogonal decompositions:** $\mathbb{R}^n = \mathrm{row}(A) \oplus \ker(A)$ and $\mathbb{R}^m = \mathrm{col}(A) \oplus \ker(A^\top)$.

**Tasks**
1. Compute all four subspaces; verify dimensions satisfy Rank–Nullity.
2. Verify the orthogonality relations numerically.
3. Decompose a vector $b$ into its column-space and left-null-space components.
4. Visualise all four subspaces for a rank-2 matrix in $\mathbb{R}^3$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import null_space, orth
np.random.seed(7)

## §1  Computing the Four Subspaces

In [ ]:
def four_subspaces(A):
    """
    Compute orthonormal bases for all four fundamental subspaces of A.

    Returns dict with keys:
      'col'      : basis of col(A)   in R^m  (m x r)
      'null'     : basis of ker(A)   in R^n  (n x (n-r))
      'row'      : basis of row(A)   in R^n  (n x r)
      'left_null': basis of ker(A^T) in R^m  (m x (m-r))
      'rank'     : r
    """
    m, n = A.shape
    r = np.linalg.matrix_rank(A)
    return {
        'col'      : orth(A),
        'null'     : null_space(A),
        'row'      : orth(A.T),
        'left_null': null_space(A.T),
        'rank'     : r,
        'm': m, 'n': n
    }


# 4x5 matrix of rank 2
A = np.array([[1., 2., 3., 4., 5.],
              [2., 4., 6., 8., 10.],
              [0., 1., 0., 1., 0.],
              [0., 2., 0., 2., 0.]], dtype=float)

S = four_subspaces(A)
print(f'm={S["m"]}, n={S["n"]}, rank r={S["rank"]}')
print(f'dim col(A)      = {S["col"].shape[1]}   (should be r = {S["rank"]})')
print(f'dim ker(A)      = {S["null"].shape[1]}   (should be n-r = {S["n"]-S["rank"]})')
print(f'dim row(A)      = {S["row"].shape[1]}   (should be r = {S["rank"]})')
print(f'dim ker(A^T)    = {S["left_null"].shape[1]}   (should be m-r = {S["m"]-S["rank"]})')
print(f'\nRank-Nullity check (n): r + (n-r) = {S["rank"]} + {S["n"]-S["rank"]} = {S["n"]} ✓')
print(f'Rank-Nullity check (m): r + (m-r) = {S["rank"]} + {S["m"]-S["rank"]} = {S["m"]} ✓')

## §2  Orthogonality Relations

- $\mathrm{row}(A) \perp \ker(A)$: every row-space vector is orthogonal to every null-space vector.
- $\mathrm{col}(A) \perp \ker(A^\top)$: every column-space vector is orthogonal to every left-null vector.

In [ ]:
# row(A) ⊥ ker(A): the matrix row^T @ null should be zero
row_null = S['row'].T @ S['null']
print('row(A)^T  @ ker(A) (max abs value, should be ~0):')
print(f'  max |entry| = {np.max(np.abs(row_null)):.2e}')

# col(A) ⊥ ker(A^T)
col_lnull = S['col'].T @ S['left_null']
print('col(A)^T  @ ker(A^T) (max abs value, should be ~0):')
print(f'  max |entry| = {np.max(np.abs(col_lnull)):.2e}')

# Spanning check: row(A) + ker(A) = R^n ?
all_n = np.hstack([S['row'], S['null']])
print(f'\nrank([row|null]) = {np.linalg.matrix_rank(all_n)} (should be n={S["n"]})')

all_m = np.hstack([S['col'], S['left_null']])
print(f'rank([col|ker(A^T)]) = {np.linalg.matrix_rank(all_m)} (should be m={S["m"]})')

## §3  Decomposing a Vector

Every $b \in \mathbb{R}^m$ decomposes uniquely as $b = b_{\mathrm{col}} + b_{\ker^\top}$ where $b_{\mathrm{col}} \in \mathrm{col}(A)$ and $b_{\ker^\top} \in \ker(A^\top)$.

The system $Ax = b$ is solvable iff $b_{\ker^\top} = 0$.

In [ ]:
def project(v, basis):
    """Orthogonal projection of v onto the column span of basis."""
    if basis.shape[1] == 0:
        return np.zeros_like(v)
    return basis @ (basis.T @ v)


# A consistent b (in col(A)) and an inconsistent one
b_in  = A[:, 0] + 3 * A[:, 2]   # in col(A)
b_out = np.array([1., 0., 1., 0.])  # random; probably not in col(A)

for b, label in [(b_in, 'b in col(A)'), (b_out, 'b not in col(A)')]:
    b_col  = project(b, S['col'])
    b_lnul = b - b_col
    solvable = np.linalg.norm(b_lnul) < 1e-8
    print(f'--- {label} ---')
    print(f'  b             = {np.round(b, 3)}')
    print(f'  b_col         = {np.round(b_col, 3)}')
    print(f'  b_ker(A^T)    = {np.round(b_lnul, 3)}')
    print(f'  ||b_ker(A^T)|| = {np.linalg.norm(b_lnul):.2e}')
    print(f'  Ax=b solvable: {solvable}')
    print()

## §4  Visualisation in $\mathbb{R}^3$

For a $3 \times 3$ rank-2 matrix, the four subspaces in $\mathbb{R}^3$ are:
- $\mathrm{col}(A)$: 2-dimensional plane in $\mathbb{R}^3$
- $\ker(A^\top)$: 1-dimensional line (the normal to the above plane)
- $\mathrm{row}(A)$: 2-dimensional plane in $\mathbb{R}^3$
- $\ker(A)$: 1-dimensional line (normal to row-space plane)

In [ ]:
B = np.array([[1., 2., 0.],
              [0., 1., 1.],
              [1., 3., 1.]], dtype=float)   # rank 2: rows 1 and 2 are independent; row3=row1+row2

SB = four_subspaces(B)
print(f'rank = {SB["rank"]}')
print('ker(B)   = span of:', SB['null'].T)
print('ker(B^T) = span of:', SB['left_null'].T)

fig = plt.figure(figsize=(12, 5))

def draw_plane(ax, normal, color, alpha=0.2, size=2.5):
    """Draw a plane with given normal through the origin."""
    n = normal / np.linalg.norm(normal)
    # Find two vectors in the plane
    if abs(n[0]) < 0.9:
        v1 = np.cross(n, [1,0,0]); v1 /= np.linalg.norm(v1)
    else:
        v1 = np.cross(n, [0,1,0]); v1 /= np.linalg.norm(v1)
    v2 = np.cross(n, v1)
    s, t = np.mgrid[-size:size:20j, -size:size:20j]
    pts = np.einsum('ij,k->ijk', s, v1) + np.einsum('ij,k->ijk', t, v2)
    ax.plot_surface(pts[:,:,0], pts[:,:,1], pts[:,:,2], color=color, alpha=alpha)

for ax_idx, (subsp, title) in enumerate([
    ({'plane': SB['col'], 'line': SB['left_null']}, 'col(B) and ker(B$^T$) in $\\mathbb{R}^3$'),
    ({'plane': SB['row'], 'line': SB['null']}, 'row(B) and ker(B) in $\\mathbb{R}^3$')
]):
    ax = fig.add_subplot(1, 2, ax_idx+1, projection='3d')
    normal_vec = subsp['line'][:, 0]
    draw_plane(ax, normal_vec, 'steelblue', alpha=0.3)
    # draw the normal (line through origin)
    t_vals = np.linspace(-2.5, 2.5, 100)
    lpts = np.outer(t_vals, normal_vec)
    ax.plot(lpts[:,0], lpts[:,1], lpts[:,2], 'r-', lw=3, label='kernel (line)')
    ax.set_xlim(-2,2); ax.set_ylim(-2,2); ax.set_zlim(-2,2)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_title(title); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# Verify A^T A and A have the same null space
print('ker(A^T A) == ker(A)?')
K_A    = null_space(B)
K_AtA  = null_space(B.T @ B)
print(f'  dim ker(A)     = {K_A.shape[1]}')
print(f'  dim ker(A^T A) = {K_AtA.shape[1]}')
# Check that K_A and K_AtA span the same subspace
combined = np.hstack([K_A, K_AtA])
print(f'  rank([ker(A)|ker(A^T A)]) = {np.linalg.matrix_rank(combined)} (should equal dim ker = {K_A.shape[1]})')